# 0. Message to Teammates

This notebook contains the code we will use to develop and train the **TinyML weather prediction model** for our capstone project.

I added explanations throughout the notebook so that everyone on the team can follow the development process, even without prior experience in machine learning, TensorFlow, or Python. Each major section explains **what we are doing, why we are doing it, and how it contributes to the final weather prediction device**.

The overall process will involve collecting historical weather data, preparing and analyzing the data, training and evaluating a machine learning model, and eventually converting the trained model so that it can run directly on our **ESP32**.

Please feel free to experiment with the notebook and contribute. If something is unfamiliar, then the explanations here, along with online resources, YouTube, and AI tools, can be used to learn more about each step.

The goal is for this notebook to document not only the final model, but also **how we developed it and the engineering decisions we made along the way (especially to showcase to the TA/Professor).**

## 1. Setting Up the Python Environment

Before working with the weather data, we have to load the Python libraries that will be used throughout the project.

- **TensorFlow** — Used to create and train the machine learning model.
- **Pandas** — Used to organize and manipulate weather data in tables.
- **NumPy** — Used for numerical calculations and arrays.
- **Matplotlib** — Used to create graphs and visualize the weather data.
- **Scikit-learn** — Provides useful tools for preparing data and evaluating the model.

The code below also prints the version of each library. This helps us make sure everyone on the team is using compatible software versions.

The GPU check tells us whether TensorFlow can use a dedicated graphics card. A GPU is not required for this project because our TinyML model will be relatively small and is ultimately designed to run on an ESP32. (so... just don't worry about it, its just a checker for me)

In [8]:
# Import dependencies and libraries
import sys
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib
import sklearn

# Validation check to see if versions match up
print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU")) 

# NOTE: Do not worry if GPU list is empty, the TinyML model is going to be small enough to run on a CPU instead of needing a GPU anyway


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
TensorFlow: 2.21.0
Pandas: 3.0.6
NumPy: 2.4.6
scikit-learn: 1.9.1
GPUs: []


## 2. Downloading Historical Weather Data

Before our physical weather sensors are available, we need real weather measurements that we can use to develop and train the machine learning model.

For our initial dataset, we are downloading historical observations from **University Park Airport (UNV)**. These observations contain measurements similar to what our final ESP32 weather device will collect, including temperature, humidity, and atmospheric pressure.

We are downloading approximately **2015–2025** so the model will eventually have several years of different seasons and weather conditions to learn from.

The basic process is:

**Iowa Environmental Mesonet → Request UNV weather observations → Download data → Save as a CSV file**

A **CSV file** is essentially a spreadsheet stored as a text file. Each row represents a weather observation and each column represents a measurement such as temperature or wind speed.

The code also checks whether we already downloaded the file. If it exists, it skips the download so we do not download the same 26 MB dataset every time the notebook runs.

In [ ]:
import requests
from pathlib import Path

STATION = "UNV"          # University Park Airport (KUNV)
START = "2015-01-01T00:00:00Z"
END   = "2026-01-01T00:00:00Z"

raw_path = Path("../data/raw/UNV_asos_2015_2025.csv")   # ../ because the notebook lives in notebooks/

url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

# Tell the weather database which station and time period we want
params = {
    "station": STATION,
    "data": "all",
    "sts": START,
    "ets": END,
    "tz": "Etc/UTC",
    "format": "onlycomma",
    "missing": "empty",
    "trace": "empty",
    "report_type": [3, 4],
}

# Only downloads the dataset if it is not already saved on this computer
if raw_path.exists():
    print("Already downloaded, skipping.")
else:
    r = requests.get(url, params=params, timeout=300)
    r.raise_for_status()
    raw_path.write_bytes(r.content)
    print(f"Saved {raw_path} ({raw_path.stat().st_size / 1e6:.1f} MB)")

Already downloaded, skipping.


## 3. Loading the Weather Data

The historical weather observations are currently stored in a CSV file on the computer.

Pandas loads this file into something called a **DataFrame**. A DataFrame can be thought of as a spreadsheet inside Python:

- Each **row** represents one weather observation.
- Each **column** represents a variable such as temperature, humidity, pressure, or wind speed.

After loading the file, we check its size and display the first few observations to make sure the data loaded correctly.

In [13]:
import pandas as pd

df = pd.read_csv(raw_path, low_memory=False)

print("Number of weather observations:", df.shape[0])
print("Number of available variables:", df.shape[1])
print(df.shape)

df.head()

Number of weather observations: 159315
Number of available variables: 30
(159315, 30)


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,wxcodes,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth
0,UNV,2015-01-01 00:53,17.60,3.20,52.60,240.0,8.0,0.0,30.14,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.37,KUNV 010053Z 24008KT 10SM SKC M08/M16 A3014,NaN
1,UNV,2015-01-01 01:53,19.40,3.20,48.69,220.0,7.0,0.0,30.12,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.42,KUNV 010153Z 22007KT 10SM SKC M07/M16 A3012,NaN
2,UNV,2015-01-01 02:55,19.58,4.82,52.05,220.0,8.0,0.0,30.12,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.81,KUNV 010255Z AUTO 22008KT 10SM M07/M15 A3012 R...,NaN
3,UNV,2015-01-01 03:15,20.30,5.00,50.90,230.0,11.0,0.0,30.12,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.68,KUNV 010315Z AUTO 23011G14KT 10SM CLR M06/M15 ...,NaN
4,UNV,2015-01-01 03:35,20.66,5.18,50.54,240.0,13.0,0.0,30.12,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.04,KUNV 010335Z AUTO 24013G18KT 10SM CLR M06/M15 ...,NaN


### Previewing the Dataset

Below are the first few weather observations in the dataset.

Some important columns include:

| Column | Meaning |
|---|---|
| `valid` | Date and time of the observation |
| `tmpf` | Air temperature in °F |
| `dwpf` | Dew point temperature in °F |
| `relh` | Relative humidity (%) |
| `drct` | Wind direction in degrees |
| `sknt` | Wind speed in knots |
| `p01i` | Precipitation |
| `alti` | Atmospheric pressure (altimeter setting) |

Not every available measurement will necessarily be used by our machine learning model. Later, we will select the measurements that are useful and that can realistically be measured by our ESP32 weather station.

`NaN` means that a particular measurement was not available for that observation. We will handle missing measurements during the data cleaning process when we get to it.

## 4. Inspecting the Available Measurements

Before deciding what information the machine learning model should use, we need to understand what measurements are available in the dataset.

The following command displays the names of all columns provided by the weather database.

This allows us to compare the historical dataset with the sensors planned for our physical weather station and decide which measurements should become inputs to the TinyML model.

In [11]:
print(df.columns.tolist())

['station', 'valid', 'tmpf', 'dwpf', 'relh', 'drct', 'sknt', 'p01i', 'alti', 'mslp', 'vsby', 'gust', 'skyc1', 'skyc2', 'skyc3', 'skyc4', 'skyl1', 'skyl2', 'skyl3', 'skyl4', 'wxcodes', 'ice_accretion_1hr', 'ice_accretion_3hr', 'ice_accretion_6hr', 'peak_wind_gust', 'peak_wind_drct', 'peak_wind_time', 'feel', 'metar', 'snowdepth']
